# F1 Car Detection and Tracking Using YOLOv8 and DeepSORT
**Author:** Madhavan Panneerselvam Kumar &emsp;|&emsp; **Date:** June 2026

---

## Abstract

This notebook presents a complete end-to-end pipeline for detecting and tracking Formula 1 cars in broadcast race footage. The system uses **YOLOv8** (object detection) combined with **DeepSORT** (multi-object tracking) and runs entirely on CPU without any domain-specific fine-tuning.

Key results on a 60-second, 1920×1080 clip at 50 fps:

| Metric | Value |
|--------|-------|
| Frames processed | 3,000 |
| Total detections | 4,397 |
| Unique track IDs | **25** |
| Average track length | **175.9 frames** (3.5 s) |
| Short ghost tracks (<10 frames) | **0%** |
| ID switches (pseudo-GT) | **1** |

A critical finding during development — setting `embedder_gpu=False` in DeepSORT — reduced fragmented track IDs from **143 → 25** and increased average track length from **13 → 175.9 frames**.

---

## Table of Contents
1. [Setup & Dependencies](#1-setup)
2. [Configuration](#2-configuration)
3. [Utility Functions](#3-utilities)
4. [Detection Module — YOLOv8](#4-detection)
5. [Tracking Module — DeepSORT](#5-tracking)
6. [Visualization Module](#6-visualization)
7. [Evaluation Module](#7-evaluation)
8. [Pseudo-Ground-Truth Generation](#8-pseudo-gt)
9. [Full Pipeline Execution](#9-pipeline)
10. [Model Comparison — YOLOv8n vs YOLOv8s](#10-comparison)
11. [Analysis & Charts](#11-analysis)
12. [Results & Conclusion](#12-results)

---
## 1. Setup & Dependencies <a id="1-setup"></a>

The pipeline depends on the following libraries:

| Library | Purpose |
|---------|--------|
| `ultralytics` | YOLOv8 object detection |
| `deep-sort-realtime` | DeepSORT multi-object tracker |
| `motmetrics` | MOTA / IDF1 evaluation metrics |
| `opencv-python` | Video I/O and frame processing |
| `matplotlib` | Analysis plots |
| `tqdm` | Progress bars |

> **Note:** `numpy<2` is required for OpenCV 4.x and Ultralytics compatibility.

In [ ]:
import sys

# Install all required packages
!{sys.executable} -m pip install -q ultralytics deep-sort-realtime motmetrics \
                                     tqdm opencv-python matplotlib
!{sys.executable} -m pip install -q 'numpy<2'   # OpenCV + Ultralytics compatibility

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f'PyTorch {torch.__version__}  |  Device: {device}')
print('All dependencies installed.')

---
## 2. Configuration <a id="2-configuration"></a>

All paths and hyperparameters are defined here. Change `INPUT_VIDEO` to point to your clip before running the pipeline.

### Parameter Rationale

| Parameter | Value | Why |
|-----------|-------|-----|
| `CONF` | `0.25` | Lower threshold improves recall on motion-blurred cars at 300 km/h |
| `MAX_AGE` | `70` | ~1.4 s at 50 fps — long enough to survive typical F1 camera cuts |
| `N_INIT` | `5` | Requires 5 consecutive hits before confirming a track; eliminates ghost tracks from motion blur |
| `MAX_COSINE_DIST` | `0.3` | Stricter appearance matching; prevents ID swaps between visually similar liveries |

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_VIDEO  = 'f1_trimmed.mp4'       # ← change to your video clip path
OUTPUT_VIDEO = 'result.mp4'           # annotated output video
GT_OUTPUT    = 'annotations/gt.txt'   # pseudo-ground-truth file

# ── Detection ──────────────────────────────────────────────────────────────
MODEL_NAME = 'yolov8n.pt'   # yolov8n.pt (fast) | yolov8s.pt | yolov8m.pt (accurate)
CONF       = 0.25           # detection confidence threshold

# ── Tracker ────────────────────────────────────────────────────────────────
MAX_AGE         = 70    # frames a track survives without a detection
N_INIT          = 5     # consecutive hits required to confirm a new track
MAX_COSINE_DIST = 0.3   # ReID appearance similarity threshold

# ── Optional ───────────────────────────────────────────────────────────────
MAX_FRAMES = None   # set to e.g. 300 for a quick test run; None = full video

print('Configuration loaded.')
print(f'  Input  : {INPUT_VIDEO}')
print(f'  Model  : {MODEL_NAME}  |  Conf: {CONF}')
print(f'  Tracker: max_age={MAX_AGE}, n_init={N_INIT}, cosine={MAX_COSINE_DIST}')

---
## 3. Utility Functions <a id="3-utilities"></a>

Shared helper functions used across the pipeline:

- **`get_video_info()`** — reads FPS, resolution, and frame count from a video file using OpenCV
- **`xyxy_to_xywh()` / `xywh_to_xyxy()`** — converts bounding box formats (DeepSORT uses `[x, y, w, h]`; YOLO outputs `[x1, y1, x2, y2]`)
- **`ensure_dir()`** — creates output directories if they don't exist

In [ ]:
import cv2


def get_video_info(video_path: str) -> dict:
    # Return FPS, resolution, frame count, and duration for a video file.
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open video: {video_path}')
    info = {
        'fps':          cap.get(cv2.CAP_PROP_FPS),
        'total_frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'width':        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height':       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'duration_sec': cap.get(cv2.CAP_PROP_FRAME_COUNT) / max(cap.get(cv2.CAP_PROP_FPS), 1),
    }
    cap.release()
    return info


def xyxy_to_xywh(bbox: list) -> list:
    # Convert [x1, y1, x2, y2] to [x, y, w, h].
    x1, y1, x2, y2 = bbox
    return [x1, y1, x2 - x1, y2 - y1]


def xywh_to_xyxy(bbox: list) -> list:
    # Convert [x, y, w, h] to [x1, y1, x2, y2].
    x, y, w, h = bbox
    return [x, y, x + w, y + h]


def ensure_dir(path: str) -> None:
    # Create a directory (and all parents) if it does not already exist.
    os.makedirs(path, exist_ok=True)


# Verify video is accessible
if os.path.exists(INPUT_VIDEO):
    info = get_video_info(INPUT_VIDEO)
    print(f"Video : {info['width']}x{info['height']}  "
          f"{info['total_frames']} frames @ {info['fps']:.1f} fps  "
          f"({info['duration_sec']:.1f} s)")
else:
    print(f"[!] '{INPUT_VIDEO}' not found — update INPUT_VIDEO in Section 2.")

---
## 4. Detection Module — YOLOv8 <a id="4-detection"></a>

### Background

YOLOv8 (Ultralytics, 2023) is a single-stage object detector with a **CSPDarknet backbone**, **PANet neck**, and an **anchor-free detection head**. It is pre-trained on the COCO dataset (80 classes). For this project, detection is restricted to **class index 2 ("car")** — no domain-specific fine-tuning is required.

### Why `conf = 0.25`?

F1 cars travelling at 300 km/h appear severely motion-blurred in broadcast footage. At the default confidence of 0.5, many genuine car detections are rejected. Lowering the threshold to 0.25 improves recall. The downstream DeepSORT tracker acts as a natural filter: a detection must be matched `n_init = 5` consecutive frames before a track is confirmed, suppressing one-off false positives.

### Pipeline Flow

```
Input frame (1920x1080 BGR)
        ↓
  CSPDarknet backbone  →  feature maps at 3 scales
        ↓
  PANet neck           →  multi-scale feature fusion
        ↓
  Anchor-free head     →  class scores + bounding boxes
        ↓
  Filter: class=2, conf >= 0.25
        ↓
  Output: [{bbox:[x1,y1,x2,y2], conf, class_id}, ...]
```

In [ ]:
from ultralytics import YOLO
import numpy as np


class CarDetector:
    # Wraps YOLOv8 to return car detections (COCO class 2) for a single BGR frame.
    # No fine-tuning required — the COCO-pretrained model generalises to F1 footage.

    CAR_CLASS_ID = 2   # COCO class index for 'car'

    def __init__(self, model_name: str = 'yolov8n.pt',
                 conf_threshold: float = 0.25, device: str = 'cpu'):
        # model_name:     YOLOv8 weight file. Auto-downloads on first use.
        #                 yolov8n.pt (6 MB, fast) | yolov8s.pt (22 MB) | yolov8m.pt (50 MB)
        # conf_threshold: Minimum confidence. 0.25 maximises recall on blurred cars.
        # device:         'cpu' for CPU-only hosts; 'cuda' if GPU is available.
        self.model  = YOLO(model_name)
        self.conf   = conf_threshold
        self.device = device

    def detect(self, frame: np.ndarray) -> list:
        # Run detection on a single BGR frame.
        # Returns: [{bbox:[x1,y1,x2,y2], conf:float, class_id:int}, ...]
        results = self.model.predict(
            source=frame,
            conf=self.conf,
            classes=[self.CAR_CLASS_ID],
            device=self.device,
            verbose=False,
        )
        detections = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = float(box.conf[0].cpu())
            detections.append({
                'bbox':     [x1, y1, x2, y2],
                'conf':     conf,
                'class_id': self.CAR_CLASS_ID,
            })
        return detections


print('CarDetector class defined.')

---
## 5. Tracking Module — DeepSORT <a id="5-tracking"></a>

### Background

**DeepSORT** (Wojke et al., ICIP 2017) extends the SORT tracker with a deep appearance descriptor. On each frame it:

1. **Predicts** each existing track's next position using a **Kalman filter**
2. **Embeds** each detection crop into a 128-D feature vector using **MobileNet**
3. **Matches** detections to tracks via a **cost matrix** combining Mahalanobis distance (position) and cosine distance (appearance)
4. **Assigns** the optimal pairing using the **Hungarian algorithm**
5. **Confirms** a track after `n_init` consecutive hits; **deletes** it after `max_age` missed frames

### Critical Bug Fix: `embedder_gpu=False`

The `deep-sort-realtime` library defaults to `embedder_gpu=True`. On a machine without a GPU this **silently produces all-zero appearance embeddings**, degrading re-identification to position-only matching. Every time a car briefly leaves the detection window, the tracker creates a brand-new ID instead of re-associating.

| | Before fix | After fix |
|--|---|---|
| Unique track IDs | **143** | **25** |
| Avg track length | 13 frames | **175.9 frames** |
| Ghost tracks | many | **0** |

In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort


class CarTracker:
    # DeepSORT wrapper for persistent multi-object tracking across video frames.
    # Each confirmed track gets a stable integer ID that persists through brief
    # occlusions and camera cuts (up to max_age frames without detection).
    #
    # CRITICAL: embedder_gpu=False is required on CPU hosts.
    # The library default (True) silently produces invalid embeddings, inflating
    # unique IDs by 5.7x and reducing avg track length from 175.9 -> 13 frames.

    def __init__(self, max_age: int = 70, n_init: int = 5,
                 max_cosine_distance: float = 0.3,
                 embedder: str = 'mobilenet', half: bool = False):
        # max_age:             Frames a track survives without a detection.
        #                      70 ~= 1.4 s at 50 fps; covers F1 camera cuts.
        # n_init:              Consecutive detections to confirm a track.
        #                      5 eliminates ghost tracks from motion-blur FPs.
        # max_cosine_distance: Appearance threshold for re-identification.
        #                      0.3 (strict) prevents ID swaps on similar liveries.
        # embedder:            'mobilenet' (fast) or 'clip_RN50' (more accurate).
        # half:                FP16 inference — only useful on GPU.
        self.tracker = DeepSort(
            max_age=max_age,
            n_init=n_init,
            max_cosine_distance=max_cosine_distance,
            embedder=embedder,
            half=half,
            embedder_gpu=False,   # MUST be False on CPU
        )

    def update(self, detections: list, frame: np.ndarray) -> list:
        # Feed one frame's detections into the tracker.
        # Returns confirmed tracks: [{track_id, bbox:[x1,y1,x2,y2], conf}, ...]
        raw = [
            (
                [d['bbox'][0], d['bbox'][1],
                 d['bbox'][2] - d['bbox'][0],
                 d['bbox'][3] - d['bbox'][1]],
                d['conf'],
                d['class_id'],
            )
            for d in detections
        ]
        tracks = self.tracker.update_tracks(raw, frame=frame)
        confirmed = []
        for t in tracks:
            if not t.is_confirmed():
                continue
            x1, y1, x2, y2 = map(int, t.to_ltrb())
            confirmed.append({
                'track_id': t.track_id,
                'bbox':     [x1, y1, x2, y2],
                'conf':     t.det_conf if t.det_conf else 0.0,
            })
        return confirmed


print('CarTracker class defined.')

---
## 6. Visualization Module <a id="6-visualization"></a>

Each confirmed track is rendered with:
- A **colored bounding box** — color is deterministic per `track_id` (seeded RNG ensures the same car always gets the same color throughout the video)
- A **filled label** showing `Car #ID` in white text

The `write_video()` function encodes the annotated frame list to an MP4 using OpenCV's `mp4v` codec at the original source frame rate.

In [ ]:
import cv2
import numpy as np


def _color_for_id(track_id: int) -> tuple:
    # Return a deterministic BGR color for a given track ID.
    np.random.seed(int(track_id) * 37)
    return tuple(int(x) for x in np.random.randint(80, 255, 3))


def draw_tracks(frame: np.ndarray, tracks: list) -> np.ndarray:
    # Overlay bounding boxes and Car #IDs onto a single frame.
    # Each track ID gets a unique persistent color.
    # Returns an annotated copy (original frame is not modified).
    out = frame.copy()
    for t in tracks:
        x1, y1, x2, y2 = t['bbox']
        color = _color_for_id(t['track_id'])
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        label = f"Car #{t['track_id']}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return out


def write_video(frames: list, output_path: str, fps: float) -> None:
    # Encode a list of annotated BGR frames to an MP4 file.
    if not frames:
        raise ValueError('No frames to write.')
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for f in frames:
        writer.write(f)
    writer.release()
    print(f'Video saved -> {output_path}  ({len(frames)} frames @ {fps:.1f} fps)')


print('draw_tracks / write_video defined.')

---
## 7. Evaluation Module <a id="7-evaluation"></a>

Since manual ground-truth annotations are unavailable, the project uses **two complementary evaluation strategies**:

### Strategy 1 — Track Stability Metrics (no GT required)
Measures tracking quality by analysing track lifespans directly:

| Metric | Description | Ideal value |
|--------|-------------|-------------|
| `avg_track_length` | Mean frames a track ID is active | As high as possible |
| `fragmentation` | Unique IDs / max simultaneous cars | ~1.0 |
| `short_track_ratio` | Fraction of tracks < 10 frames (ghost tracks) | 0.0 |
| `detection_rate` | Fraction of frames with at least 1 detection | Depends on footage |

### Strategy 2 — Pseudo-GT MOT Metrics
Uses `py-motmetrics` to compute standard **MOTA** and **IDF1** against a pseudo-ground-truth:

```
MOTA = 1 - (FP + FN + IDSW) / |GT|
```

> **Limitation:** The oracle (conf=0.5) and predictor (conf=0.25) use different confidence thresholds. Lower-confidence detections are counted as false positives, inflating FP and suppressing MOTA. The **ID switch count of 1** is the most reliable metric.

In [ ]:
import motmetrics as mm
import numpy as np


# ── Load MOT-format ground truth ──────────────────────────────────────────
def load_mot_gt(gt_txt_path: str) -> dict:
    # Parse a MOT-format GT file: frame,id,x,y,w,h,conf,-1,-1,-1
    # Returns: {frame_id: [{bbox:[x1,y1,x2,y2], gt_id:int}, ...]}
    gt_frames = {}
    with open(gt_txt_path) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 6:
                continue
            frame_id = int(parts[0])
            gt_id    = int(parts[1])
            x, y, w, h = int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])
            gt_frames.setdefault(frame_id, []).append(
                {'bbox': [x, y, x + w, y + h], 'gt_id': gt_id})
    return gt_frames


# ── MOTA / IDF1 via py-motmetrics ─────────────────────────────────────────
def compute_metrics_with_gt(gt_frames: dict, pred_frames: dict) -> dict:
    # Compute MOTA, IDF1, ID switches, misses, and false positives.
    # Matching is done by IoU >= 0.5 between GT and predicted bounding boxes.
    acc = mm.MOTAccumulator(auto_id=True)
    for frame_id in sorted(set(list(gt_frames.keys()) + list(pred_frames.keys()))):
        gts      = gt_frames.get(frame_id, [])
        preds    = pred_frames.get(frame_id, [])
        gt_ids   = [g['gt_id']    for g in gts]
        pred_ids = [p['track_id'] for p in preds]
        if gts and preds:
            dist = mm.distances.iou_matrix(
                [g['bbox'] for g in gts],
                [p['bbox'] for p in preds],
                max_iou=0.5,
            )
        else:
            dist = np.empty((len(gt_ids), len(pred_ids)))
        acc.update(gt_ids, pred_ids, dist)
    mh      = mm.metrics.create()
    summary = mh.compute(acc, metrics=[
        'mota', 'idf1', 'num_switches', 'num_misses', 'num_false_positives'])
    return {
        'MOTA':        round(float(summary['mota'].iloc[0]) * 100, 2),
        'IDF1':        round(float(summary['idf1'].iloc[0]) * 100, 2),
        'ID_Switches': int(summary['num_switches'].iloc[0]),
        'Misses':      int(summary['num_misses'].iloc[0]),
        'FP':          int(summary['num_false_positives'].iloc[0]),
    }


# ── Basic statistics (no GT needed) ───────────────────────────────────────
def count_basic_stats(pred_frames: dict) -> dict:
    # Compute detection statistics directly from tracked predictions.
    all_ids    = set()
    max_sim    = 0
    total_dets = 0
    for tracks in pred_frames.values():
        all_ids.update(t['track_id'] for t in tracks)
        max_sim    = max(max_sim, len(tracks))
        total_dets += len(tracks)
    n = len(pred_frames)
    return {
        'total_unique_tracks':   len(all_ids),
        'max_simultaneous_cars': max_sim,
        'avg_simultaneous_cars': round(total_dets / n, 2) if n else 0,
        'total_frames':          n,
        'total_detections':      total_dets,
    }


# ── Track stability metrics (no GT needed) ────────────────────────────────
def compute_track_stability(pred_frames: dict) -> dict:
    # Measure tracking quality without GT by analysing track lifespans.
    track_frames     = {}
    frames_with_dets = 0
    for frame_id, tracks in sorted(pred_frames.items()):
        if tracks:
            frames_with_dets += 1
        for t in tracks:
            track_frames.setdefault(t['track_id'], []).append(frame_id)
    if not track_frames:
        return {'avg_track_length': 0, 'fragmentation': 0,
                'short_track_ratio': 0, 'detection_rate': 0}
    lengths = [len(v) for v in track_frames.values()]
    max_sim = max(len(pred_frames[f]) for f in pred_frames) if pred_frames else 1
    return {
        'avg_track_length':  round(sum(lengths) / len(lengths), 1),
        'fragmentation':     round(len(track_frames) / max(max_sim, 1), 2),
        'short_track_ratio': round(sum(1 for l in lengths if l < 10) / len(lengths), 2),
        'detection_rate':    round(frames_with_dets / max(len(pred_frames), 1), 2),
    }


print('Evaluation functions defined.')

---
## 8. Pseudo-Ground-Truth Generation <a id="8-pseudo-gt"></a>

Since manual frame-level annotation (e.g., via CVAT) is time-prohibitive for a 3,000-frame clip, this project uses a **pseudo-GT oracle**:

1. Run **YOLOv8n at conf = 0.5** — only high-confidence, unambiguous detections
2. Track with **DeepSORT** (same tracker) to assign stable object IDs
3. Save in **MOT format** for use with `py-motmetrics`

**Result:** 802 annotations, 9 unique GT track IDs saved to `annotations/gt.txt`.

> **Evaluation caveat:** Because the oracle uses conf=0.5 and the predictor uses conf=0.25, all lower-confidence detections are counted as false positives against the GT. This inflates FP and suppresses MOTA — it does *not* reflect true tracking quality. The **1 ID switch** is the reliable metric.

In [ ]:
def generate_gt(
    input_video: str  = INPUT_VIDEO,
    output_gt:   str  = GT_OUTPUT,
    conf:        float = 0.5,
    max_frames:  int  = None,
) -> None:
    # Generate pseudo-GT annotations in MOT format.
    # Uses YOLOv8n at high confidence (conf=0.5) + DeepSORT oracle tracking.
    # Output format: frame_id,object_id,x,y,w,h,1,-1,-1,-1
    from tqdm import tqdm
    print('=== Pseudo-GT Generator ===')
    print(f'  Input : {input_video}  |  Output: {output_gt}  |  Conf: {conf}')

    ensure_dir(os.path.dirname(output_gt))
    model   = YOLO('yolov8n.pt')
    tracker = CarTracker(max_age=70, n_init=3, max_cosine_distance=0.3)

    cap      = cv2.VideoCapture(input_video)
    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit    = min(total, max_frames) if max_frames else total
    gt_lines = []

    with tqdm(total=limit, unit='frame', desc='Generating GT') as pbar:
        for frame_id in range(limit):
            ret, frame = cap.read()
            if not ret:
                break
            results = model.predict(source=frame, conf=conf,
                                    classes=[2], device='cpu', verbose=False)
            dets = []
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                dets.append({'bbox': [x1, y1, x2, y2],
                             'conf': float(box.conf[0].cpu()), 'class_id': 2})
            tracks = tracker.update(dets, frame)
            for t in tracks:
                x1, y1, x2, y2 = t['bbox']
                w, h = x2 - x1, y2 - y1
                gt_lines.append(
                    f"{frame_id + 1},{t['track_id']},{x1},{y1},{w},{h},1,-1,-1,-1")
            pbar.update(1)
    cap.release()

    with open(output_gt, 'w') as f:
        f.write('\n'.join(gt_lines))

    unique_ids = len(set(int(l.split(',')[1]) for l in gt_lines))
    print(f'\nDone - {len(gt_lines)} annotations  |  {unique_ids} unique GT track IDs')
    print(f'Saved -> {output_gt}')


# Generate GT if it doesn't already exist
if not os.path.exists(GT_OUTPUT):
    generate_gt()
else:
    lines = open(GT_OUTPUT).readlines()
    ids   = len(set(int(l.split(',')[1]) for l in lines if l.strip()))
    print(f'GT already exists: {len(lines)} annotations, {ids} unique track IDs')

---
## 9. Full Pipeline Execution <a id="9-pipeline"></a>

The pipeline runs the following sequence on every frame:

```
Read frame
   -> YOLOv8 detect  (class=car, conf=0.25)
   -> DeepSORT update (Kalman + MobileNet + Hungarian)
   -> draw_tracks     (overlay boxes + IDs onto frame)
   -> store to pred_frames
```

After processing all frames:
- Annotated frames are encoded to `result.mp4`
- Basic statistics and track stability metrics are printed
- If `annotations/gt.txt` exists, MOTA / IDF1 are computed against pseudo-GT

> **Runtime:** ~15 minutes on CPU for 3,000 frames.  
> Set `MAX_FRAMES = 300` in Section 2 for a quick test (~90 seconds).

In [ ]:
def run_pipeline(
    input_video:  str   = INPUT_VIDEO,
    output_video: str   = OUTPUT_VIDEO,
    model:        str   = MODEL_NAME,
    conf:         float = CONF,
    max_frames:   int   = MAX_FRAMES,
    gt_path:      str   = None,
):
    # End-to-end F1 car detection and tracking pipeline.
    # Returns: pred_frames (dict), annotated_frames (list), fps (float)
    from tqdm import tqdm

    print('\n=== F1 Car Detection & Tracking Pipeline ===')
    print(f'  Input  : {input_video}')
    print(f'  Output : {output_video}')
    print(f'  Model  : {model}  |  Conf: {conf}')

    info  = get_video_info(input_video)
    fps   = info['fps']
    total = info['total_frames']
    print(f"  Video  : {info['width']}x{info['height']}  "
          f"{total} frames @ {fps:.1f} fps  ({info['duration_sec']:.1f} s)\n")

    print('Loading YOLOv8 detector ...')
    detector = CarDetector(model_name=model, conf_threshold=conf)
    print('Loading DeepSORT tracker ...')
    tracker  = CarTracker(max_age=MAX_AGE, n_init=N_INIT,
                          max_cosine_distance=MAX_COSINE_DIST)

    cap              = cv2.VideoCapture(input_video)
    limit            = min(total, max_frames) if max_frames else total
    pred_frames      = {}
    annotated_frames = []

    print(f'\nProcessing {limit} frames ...')
    with tqdm(total=limit, unit='frame') as pbar:
        for frame_id in range(limit):
            ret, frame = cap.read()
            if not ret:
                break
            dets   = detector.detect(frame)
            tracks = tracker.update(dets, frame)
            pred_frames[frame_id] = tracks
            annotated_frames.append(draw_tracks(frame, tracks))
            pbar.update(1)
    cap.release()

    write_video(annotated_frames, output_video, fps)

    print('\n=== Detection & Tracking Statistics ===')
    stats = count_basic_stats(pred_frames)
    labels = {
        'total_frames':          'Frames processed',
        'total_detections':      'Total detections',
        'total_unique_tracks':   'Unique track IDs',
        'max_simultaneous_cars': 'Max cars in one frame',
        'avg_simultaneous_cars': 'Avg cars per frame',
    }
    for key, label in labels.items():
        print(f'  {label:<28}: {stats[key]}')

    print('\n=== Track Stability (no GT required) ===')
    stab = compute_track_stability(pred_frames)
    stab_labels = {
        'avg_track_length':  'Avg track length (frames)',
        'fragmentation':     'Fragmentation ratio',
        'short_track_ratio': 'Short tracks <10 frames',
        'detection_rate':    'Detection rate',
    }
    for key, label in stab_labels.items():
        print(f'  {label:<28}: {stab[key]}')

    if gt_path and os.path.exists(gt_path):
        print(f'\n=== MOT Metrics (pseudo-GT) ===')
        gt_frames = load_mot_gt(gt_path)
        metrics   = compute_metrics_with_gt(gt_frames, pred_frames)
        mot_labels = {
            'MOTA': 'MOTA', 'IDF1': 'IDF1',
            'ID_Switches': 'ID Switches',
            'Misses': 'Misses (FN)', 'FP': 'False Positives',
        }
        for key, label in mot_labels.items():
            print(f'  {label:<28}: {metrics[key]}')
        print('  Note: MOTA inflated by conf threshold mismatch (see Section 7).')

    print(f'\nPipeline complete. Output: {output_video}')
    return pred_frames, annotated_frames, fps


print('run_pipeline() defined.')

### Run the Pipeline

> Estimated runtime: ~15 min for 3,000 frames on CPU.  
> For a quick test, set `MAX_FRAMES = 300` in Section 2 (~90 seconds).

In [ ]:
pred_frames, annotated_frames, fps = run_pipeline(
    gt_path=GT_OUTPUT if os.path.exists(GT_OUTPUT) else None,
)

---
## 10. Model Comparison — YOLOv8n vs YOLOv8s <a id="10-comparison"></a>

To guide model selection, both variants are benchmarked on 300 frames at the same confidence threshold (conf = 0.25).

### Results (300 frames, CPU inference)

| Model | Avg dets / frame | Max dets | Avg confidence | Speed |
|-------|-----------------|----------|----------------|-------|
| `yolov8n.pt` | **1.32** | **4** | 0.383 | **3.4 fps** |
| `yolov8s.pt` | 0.27 | 3 | 0.371 | 1.9 fps |

**Key finding:** Contrary to expectation, YOLOv8n detects ~5x more cars per frame than YOLOv8s at the same confidence threshold. YOLOv8s is more selective (higher precision, lower recall). For motion-blurred sports footage where recall is critical, and given CPU-only deployment, **YOLOv8n is the better choice** on both quality and speed grounds.

In [ ]:
import time
import matplotlib.pyplot as plt


def benchmark_model(video_path: str, model_name: str,
                    conf: float = 0.25, n_frames: int = 300) -> dict:
    # Run a model on n_frames and return detection and speed statistics.
    model     = YOLO(model_name)
    cap       = cv2.VideoCapture(video_path)
    det_counts, conf_scores, times = [], [], []
    for _ in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        results = model.predict(source=frame, conf=conf,
                                classes=[2], device='cpu', verbose=False)
        times.append(time.perf_counter() - t0)
        boxes = results[0].boxes
        det_counts.append(len(boxes))
        conf_scores.extend(float(b.conf[0].cpu()) for b in boxes)
    cap.release()
    return {
        'model':       model_name,
        'avg_dets':    round(np.mean(det_counts), 2),
        'max_dets':    int(np.max(det_counts)),
        'avg_conf':    round(np.mean(conf_scores), 3) if conf_scores else 0,
        'fps':         round(1 / np.mean(times), 1),
        'det_counts':  det_counts,
        'conf_scores': conf_scores,
    }


def run_comparison(video_path: str = INPUT_VIDEO,
                   conf: float = 0.25, n_frames: int = 300) -> list:
    # Benchmark YOLOv8n and YOLOv8s; print a comparison table and plot results.
    print(f'Benchmarking on {n_frames} frames (conf={conf}) ...')
    results = []
    for m in ['yolov8n.pt', 'yolov8s.pt']:
        print(f'  -> {m}')
        results.append(benchmark_model(video_path, m, conf, n_frames))

    print(f"\n{'Model':<14} {'Avg dets':>9} {'Max dets':>9} {'Avg conf':>9} {'FPS':>6}")
    print('-' * 52)
    for r in results:
        print(f"{r['model']:<14} {r['avg_dets']:>9} {r['max_dets']:>9} "
              f"{r['avg_conf']:>9} {r['fps']:>6}")

    colors = ['royalblue', 'darkorange']
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for r, c in zip(results, colors):
        axes[0].plot(r['det_counts'], color=c, alpha=0.8, label=r['model'], lw=1)
    axes[0].set_title('Detections Per Frame', fontweight='bold')
    axes[0].set_xlabel('Frame'); axes[0].set_ylabel('Cars detected')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    for r, c in zip(results, colors):
        if r['conf_scores']:
            axes[1].hist(r['conf_scores'], bins=25, color=c, alpha=0.6,
                         label=r['model'], edgecolor='white')
    axes[1].set_title('Confidence Score Distribution', fontweight='bold')
    axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    fps_vals = [r['fps'] for r in results]
    bars = axes[2].bar([r['model'] for r in results], fps_vals,
                       color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, fps_vals):
        axes[2].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.1,
                     f'{val} fps', ha='center', fontweight='bold')
    axes[2].set_title('Inference Speed (CPU)', fontweight='bold')
    axes[2].set_ylabel('Frames per second')
    axes[2].set_ylim(0, max(fps_vals) * 1.3)
    axes[2].grid(alpha=0.3, axis='y')

    plt.suptitle('YOLOv8n vs YOLOv8s - F1 Detection Benchmark',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('comparison_plot.png', dpi=150)
    plt.show()
    print('Saved -> comparison_plot.png')
    return results


print('run_comparison() defined.')

In [ ]:
comparison_results = run_comparison(n_frames=300)

---
## 11. Analysis & Charts <a id="11-analysis"></a>

Two visualizations are produced after the pipeline run:

**`track_analysis.png`** — Two-panel chart:
- *Top:* Cars detected per frame over time — shows the episodic detection pattern caused by broadcast camera cuts
- *Bottom:* Track length histogram — confirms all 25 tracks are well above the 10-frame ghost-track threshold (avg = 175.9 frames)

**`annotated_frames.png`** — Five sample frames sampled uniformly at 5%, 25%, 50%, 75%, and 95% through the video, with bounding boxes and persistent Car #IDs overlaid

In [ ]:
import matplotlib.pyplot as plt


def save_track_analysis(pred_frames: dict, fps: float):
    # Cars-per-frame timeline + track length distribution histogram.
    # Saves to track_analysis.png.
    frame_ids      = sorted(pred_frames.keys())
    cars_per_frame = [len(pred_frames[f]) for f in frame_ids]
    times          = [f / fps for f in frame_ids]
    stats          = count_basic_stats(pred_frames)
    stability      = compute_track_stability(pred_frames)

    track_map = {}
    for tracks in pred_frames.values():
        for t in tracks:
            track_map[t['track_id']] = track_map.get(t['track_id'], 0) + 1
    lengths = list(track_map.values())

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    axes[0].fill_between(times, cars_per_frame, alpha=0.35, color='royalblue')
    axes[0].plot(times, cars_per_frame, color='royalblue', lw=0.9)
    axes[0].axhline(stats['avg_simultaneous_cars'], color='red', ls='--', lw=1.2,
                    label=f"avg = {stats['avg_simultaneous_cars']} cars / frame")
    axes[0].set_xlabel('Time (s)', fontsize=11)
    axes[0].set_ylabel('Cars detected', fontsize=11)
    axes[0].set_title('Cars Detected Per Frame Over Time', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

    axes[1].hist(lengths, bins=min(30, len(lengths)),
                 color='darkorange', edgecolor='white', alpha=0.85)
    axes[1].axvline(stability['avg_track_length'], color='red', ls='--', lw=1.2,
                    label=f"avg = {stability['avg_track_length']} frames")
    axes[1].set_xlabel('Track length (frames)', fontsize=11)
    axes[1].set_ylabel('Number of tracks', fontsize=11)
    axes[1].set_title(
        f"Track Length Distribution  -  {stats['total_unique_tracks']} unique IDs  "
        f"| fragmentation = {stability['fragmentation']}",
        fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('track_analysis.png', dpi=150)
    plt.show()
    print('Saved -> track_analysis.png')
    return stats, stability


def save_sample_frames(annotated_frames: list, pred_frames: dict, fps: float) -> None:
    # Display 5 annotated sample frames spread uniformly across the video.
    # Saves to annotated_frames.png.
    n          = len(annotated_frames)
    sample_ids = [int(n * p) for p in [0.05, 0.25, 0.50, 0.75, 0.95]]
    fig, axes  = plt.subplots(1, 5, figsize=(22, 4))
    for ax, fid in zip(axes, sample_ids):
        rgb    = cv2.cvtColor(annotated_frames[fid], cv2.COLOR_BGR2RGB)
        n_cars = len(pred_frames.get(fid, []))
        ax.imshow(rgb)
        ax.set_title(f"t = {fid / fps:.1f} s\n"
                     f"{n_cars} car{'s' if n_cars != 1 else ''}", fontsize=10)
        ax.axis('off')
    plt.suptitle('Sample Annotated Frames - YOLOv8n + DeepSORT',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('annotated_frames.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> annotated_frames.png')


print('Analysis functions defined.')

### Generate Analysis Charts

In [ ]:
# Requires pred_frames and annotated_frames from Section 9
stats, stability = save_track_analysis(pred_frames, fps)
save_sample_frames(annotated_frames, pred_frames, fps)

---
## 12. Results & Conclusion <a id="12-results"></a>

### Final Metrics Summary

In [ ]:
from IPython.display import Image, display

# Print consolidated results
print('=' * 55)
print('  FINAL RESULTS - F1 Car Tracking Pipeline')
print('=' * 55)
all_metrics = {**count_basic_stats(pred_frames), **compute_track_stability(pred_frames)}
for k, v in all_metrics.items():
    print(f'  {k:<30}: {v}')

# Display saved charts inline
for img_file in ['track_analysis.png', 'annotated_frames.png', 'comparison_plot.png']:
    if os.path.exists(img_file):
        print(f'\n-- {img_file} --')
        display(Image(filename=img_file))
    else:
        print(f'[!] {img_file} not found - run Sections 9-11 first.')

---
## Conclusion

This project delivered a complete **YOLOv8 + DeepSORT** tracking pipeline for F1 broadcast footage, running on CPU without domain-specific training data.

### Key Findings

| Finding | Detail |
|---------|--------|
| **Critical bug fix** | `embedder_gpu=True` (the library default) silently invalidates MobileNet embeddings on CPU. Setting it to `False` reduced unique IDs 143 → 25 and increased avg track length 13.5× |
| **Parameter tuning** | Increasing `max_age` 30→70 and `n_init` 3→5 eliminated ghost tracks and improved survival across camera cuts |
| **Model selection** | YOLOv8n outperforms YOLOv8s for this footage — 1.32 vs 0.27 avg dets/frame and 1.8× faster on CPU |
| **Tracking continuity** | Only **1 ID switch** over 3,000 frames demonstrates near-perfect identity preservation |

### Limitations & Future Work

1. **Detection recall** — Fine-tuning YOLOv8 on a labeled F1 dataset would improve detection of oblique or extreme-speed cars
2. **Real-time performance** — GPU deployment (CUDA or Apple MPS) is needed for 50 fps real-time operation
3. **Camera-cut handling** — A shot-boundary detector could reset the tracker between cuts, reducing the fragmentation ratio toward the ideal of 1.0
4. **Ground truth** — Manual annotation of ~500 frames in CVAT would give unambiguous MOTA / IDF1 scores

### References

1. Jocher, G. et al. (2023). *YOLOv8 by Ultralytics.* https://github.com/ultralytics/ultralytics
2. Wojke, N., Bewley, A., Paulus, D. (2017). *Simple Online and Realtime Tracking with a Deep Association Metric.* ICIP 2017.
3. Bewley, A. et al. (2016). *Simple Online and Realtime Tracking.* ICIP 2016.
4. Lin, T.Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* ECCV 2014.
5. Cheind. *py-motmetrics.* https://github.com/cheind/py-motmetrics